# LAB 3 — RAG-Fusion Completo
## Aula 7 · MBA RAG & CAG Aplicados a Direito e Segurança Pública

**Objetivo:** Construir um pipeline RAG-Fusion completo: geração de sub-queries com vLLM → retrieval paralelo com asyncio no OpenSearch → Reciprocal Rank Fusion (RRF) → geração de resposta com Llama 3.1 8B.

**Tempo estimado:** 45 minutos

---
**Checklist de entrega:**
- [ ] Sub-queries geradas com vLLM (N=3)
- [ ] Retrieval paralelo com `asyncio.gather()`
- [ ] RRF implementado manualmente com fórmula visível
- [ ] Exemplo numérico do RRF documentado
- [ ] Geração final com Llama 3.1 a partir do contexto fundido

In [ ]:
!pip install -q langchain langchain-openai sentence-transformers opensearch-py pandas matplotlib
print('✅ Dependências instaladas!')

In [ ]:
import os
import asyncio
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from sentence_transformers import SentenceTransformer
from opensearchpy import AsyncOpenSearch, OpenSearch

VLLM_BASE_URL = os.getenv('VLLM_BASE_URL', 'http://localhost:8000/v1')
OPENSEARCH_HOST = os.getenv('OPENSEARCH_HOST', 'localhost')
INDEX_NAME = 'corpus_juridico_aula7'

# LLM síncrono (para geração de queries e resposta final)
llm = ChatOpenAI(
    base_url=VLLM_BASE_URL, api_key='dummy',
    model='meta-llama/Llama-3.1-8B-Instruct',
    temperature=0.5, max_tokens=1024
)

# Embeddings
print('Carregando BGE-M3...')
embeddings = SentenceTransformer('BAAI/bge-m3')
print('✅ BGE-M3 pronto!')

# OpenSearch síncrono para verificação
os_client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': 9200}],
    http_auth=('admin', os.getenv('OPENSEARCH_PASS', 'Admin123!')),
    use_ssl=False, verify_certs=False
)
print(f'✅ OpenSearch: {os_client.info()["version"]["number"]}')

## 2. Geração de Sub-Queries

In [ ]:
SUBQUERY_PROMPT = PromptTemplate(
    input_variables=['question', 'n'],
    template="""Você é um assistente jurídico especializado em RAG.
Dada a pergunta abaixo, gere {n} sub-perguntas que cubram ângulos distintos
do mesmo tema. Cada sub-pergunta deve ser autocontida e mais específica que a original.

Retorne APENAS as {n} perguntas, uma por linha, sem numeração.

Pergunta: {question}

Sub-perguntas:"""
)

def generate_sub_queries(question: str, n: int = 3) -> list[str]:
    """Gera N sub-queries para a questão principal."""
    response = llm.invoke(SUBQUERY_PROMPT.format(question=question, n=n))
    queries = [q.strip() for q in response.content.strip().split('\n') if q.strip()]
    return queries[:n]  # garantir exatamente N queries

# Teste
query_principal = 'Quais são os requisitos e procedimentos para decretação de prisão preventiva no processo penal?'
sub_queries = generate_sub_queries(query_principal, n=3)

print(f'Query principal: {query_principal}')
print('\nSub-queries geradas:')
for i, sq in enumerate(sub_queries, 1):
    print(f'  {i}. {sq}')

## 3. Reciprocal Rank Fusion (RRF) — Implementação Manual

In [ ]:
def reciprocal_rank_fusion(rankings: list[list[dict]], k: int = 60) -> list[dict]:
    """
    Implementação do Reciprocal Rank Fusion (Cormack et al., 2009).
    
    Fórmula: RRF(d) = Σ 1 / (k + rank_r(d))  para cada ranking r
    
    Parâmetros:
        rankings: lista de rankings, cada um sendo uma lista de dicionários
                  com 'id' e 'content'
        k: constante de suavização (padrão=60, recomendado por Cormack et al.)
    
    Retorna:
        Lista de documentos ordenada por score RRF decrescente
    """
    doc_scores: dict[str, float] = {}
    doc_data: dict[str, dict] = {}
    
    # Para cada ranking (uma por sub-query)
    for ranking_idx, ranking in enumerate(rankings):
        # Para cada documento em sua posição neste ranking
        for rank, doc in enumerate(ranking, start=1):  # rank começa em 1
            doc_id = doc['id']
            
            # Inicializar score se documento novo
            if doc_id not in doc_scores:
                doc_scores[doc_id] = 0.0
                doc_data[doc_id] = doc
            
            # Fórmula RRF: 1 / (k + rank)
            rrf_contribution = 1.0 / (k + rank)
            doc_scores[doc_id] += rrf_contribution
    
    # Ordenar por score RRF (maior = mais relevante)
    sorted_ids = sorted(doc_scores.keys(), key=lambda d: doc_scores[d], reverse=True)
    
    return [
        {**doc_data[doc_id], 'rrf_score': doc_scores[doc_id]}
        for doc_id in sorted_ids
    ]

print('✅ RRF implementado!')

## 4. Exemplo Numérico do RRF (Obrigatório)

In [ ]:
# ── Exemplo numérico para entender o RRF ─────────────────────────────
print('=== EXEMPLO NUMÉRICO: COMO O RRF FUNCIONA ===')
print()
print('Cenário: 3 sub-queries, cada uma retornou 4 documentos')
print()

# Rankings fictícios para exemplo didático
rankings_exemplo = [
    # Ranking da Sub-query 1
    [{'id': 'doc_A', 'content': 'Prisão preventiva: requisitos do art. 312 CPP'},
     {'id': 'doc_B', 'content': 'Medidas cautelares alternativas à prisão'},
     {'id': 'doc_C', 'content': 'Decreto prisional: fundamentação necessária'},
     {'id': 'doc_D', 'content': 'Prazo máximo de prisão preventiva'}],
    
    # Ranking da Sub-query 2
    [{'id': 'doc_B', 'content': 'Medidas cautelares alternativas à prisão'},
     {'id': 'doc_A', 'content': 'Prisão preventiva: requisitos do art. 312 CPP'},
     {'id': 'doc_E', 'content': 'Habeas corpus como remédio contra prisão ilegal'},
     {'id': 'doc_C', 'content': 'Decreto prisional: fundamentação necessária'}],
    
    # Ranking da Sub-query 3
    [{'id': 'doc_A', 'content': 'Prisão preventiva: requisitos do art. 312 CPP'},
     {'id': 'doc_C', 'content': 'Decreto prisional: fundamentação necessária'},
     {'id': 'doc_B', 'content': 'Medidas cautelares alternativas à prisão'},
     {'id': 'doc_F', 'content': 'Flagrante e conversão em preventiva'}]
]

k = 60
print(f'Constante k = {k}')
print(f'Fórmula: RRF(d) = Σ 1/(k + rank)  para cada ranking')
print()

# Calcular e exibir passo a passo
docs_info = {}
for r_idx, ranking in enumerate(rankings_exemplo, 1):
    print(f'Sub-query {r_idx}:')
    for rank, doc in enumerate(ranking, start=1):
        contribuicao = 1.0 / (k + rank)
        doc_id = doc['id']
        if doc_id not in docs_info:
            docs_info[doc_id] = {'score': 0, 'rankings': [], 'content': doc['content']}
        docs_info[doc_id]['score'] += contribuicao
        docs_info[doc_id]['rankings'].append((r_idx, rank, contribuicao))
        print(f'  Rank {rank}: {doc_id} → 1/({k}+{rank}) = {contribuicao:.6f}')
    print()

print('=== SCORES FINAIS RRF ===')
sorted_docs = sorted(docs_info.items(), key=lambda x: x[1]['score'], reverse=True)
for rank_final, (doc_id, info) in enumerate(sorted_docs, 1):
    print(f'{rank_final}º {doc_id}: score={info["score"]:.6f} | {info["content"][:50]}')

print()
print('💡 Doc_A aparece em 1º em SQ1, 2º em SQ2 e 1º em SQ3 → Score alto (consistência!)')
print('   Doc_B aparece em 2º, 1º e 3º → também recebe score alto')
print('   Doc_E e Doc_F aparecem em apenas 1 ranking → scores mais baixos')

## 5. Retrieval Paralelo com asyncio

In [ ]:
# ── Cliente OpenSearch assíncrono ─────────────────────────────────────
async_os_client = AsyncOpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': 9200}],
    http_auth=('admin', os.getenv('OPENSEARCH_PASS', 'Admin123!')),
    use_ssl=False, verify_certs=False
)

async def retrieve_async(query: str, k: int = 5) -> list[dict]:
    """Retrieval assíncrono via OpenSearch kNN."""
    q_vec = embeddings.encode(query, normalize_embeddings=True).tolist()
    
    response = await async_os_client.search(
        index=INDEX_NAME,
        body={
            'size': k,
            'query': {'knn': {'embedding': {'vector': q_vec, 'k': k}}},
            '_source': ['content', 'source']
        }
    )
    
    return [
        {
            'id': h['_id'],
            'score': h['_score'],
            'content': h['_source'].get('content', ''),
            'source': h['_source'].get('source', 'N/A')
        }
        for h in response['hits']['hits']
    ]

async def rag_fusion_pipeline(
    query: str,
    n_sub_queries: int = 3,
    k_per_query: int = 5,
    rrf_k: int = 60
) -> dict:
    """
    Pipeline RAG-Fusion completo:
    1. Gerar sub-queries
    2. Retrieval paralelo com asyncio
    3. Aplicar RRF
    4. Retornar documentos fundidos
    """
    t_total = time.time()
    
    # 1. Gerar sub-queries
    t0 = time.time()
    sub_queries = generate_sub_queries(query, n=n_sub_queries)
    all_queries = sub_queries + [query]  # incluir query original
    t_gen = time.time() - t0
    
    # 2. Retrieval PARALELO (asyncio.gather = execução simultânea)
    t0 = time.time()
    tasks = [retrieve_async(q, k=k_per_query) for q in all_queries]
    rankings = await asyncio.gather(*tasks)  # ← paralelismo real aqui!
    t_retrieval = time.time() - t0
    
    # 3. RRF
    t0 = time.time()
    fused_docs = reciprocal_rank_fusion(rankings, k=rrf_k)
    t_rrf = time.time() - t0
    
    return {
        'query_original': query,
        'sub_queries': sub_queries,
        'fused_docs': fused_docs,
        'stats': {
            'n_queries': len(all_queries),
            'docs_por_query': k_per_query,
            'docs_totais_antes_rrf': sum(len(r) for r in rankings),
            'docs_unicos_apos_rrf': len(fused_docs),
            't_geracao_s': round(t_gen, 2),
            't_retrieval_s': round(t_retrieval, 2),
            't_rrf_s': round(t_rrf, 4),
            't_total_s': round(time.time() - t_total, 2)
        }
    }

print('✅ Pipeline RAG-Fusion assíncrono definido!')

## 6. Executar RAG-Fusion em Query Jurídica Complexa

In [ ]:
# ── Query jurídica complexa multi-facetada ────────────────────────────
query_complexa = 'Quais são os requisitos e procedimentos para decretação de prisão preventiva no processo penal, incluindo medidas cautelares alternativas e prazos?'

# Executar pipeline RAG-Fusion
# Nota: no Google Colab, usar asyncio.run() ou await diretamente no notebook
resultado = await rag_fusion_pipeline(
    query=query_complexa,
    n_sub_queries=3,
    k_per_query=5,
    rrf_k=60
)

print(f'Query: {resultado["query_original"]}')
print(f'\nSub-queries geradas:')
for i, sq in enumerate(resultado['sub_queries'], 1):
    print(f'  {i}. {sq}')

print(f'\n📊 Estatísticas do Pipeline:')
stats = resultado['stats']
print(f'  Queries totais (incl. original): {stats["n_queries"]}')
print(f'  Docs por query (k): {stats["docs_por_query"]}')
print(f'  Docs antes do RRF: {stats["docs_totais_antes_rrf"]}')
print(f'  Docs únicos após RRF: {stats["docs_unicos_apos_rrf"]}')
print(f'  Tempo geração sub-queries: {stats["t_geracao_s"]}s')
print(f'  Tempo retrieval paralelo: {stats["t_retrieval_s"]}s')
print(f'  Tempo RRF: {stats["t_rrf_s"]}s')
print(f'  TEMPO TOTAL: {stats["t_total_s"]}s')

print(f'\n📄 Top 5 documentos por score RRF:')
for i, doc in enumerate(resultado['fused_docs'][:5], 1):
    print(f'  {i}. [RRF={doc["rrf_score"]:.5f}] {doc["content"][:100]}...')

## 7. Geração de Resposta Final com Llama 3.1

In [ ]:
def gerar_resposta(query: str, docs_fundidos: list, top_k: int = 5) -> str:
    """
    Gera resposta final usando os documentos fundidos pelo RRF.
    """
    # Construir contexto com top-K documentos do RRF
    contexto = '\n\n'.join([
        f'[Documento {i+1} | RRF Score: {doc["rrf_score"]:.4f}]\n{doc["content"]}'
        for i, doc in enumerate(docs_fundidos[:top_k])
    ])
    
    PROMPT_GERACAO = f"""Você é um assistente jurídico especializado em Direito Processual Penal brasileiro.
Use EXCLUSIVAMENTE os documentos abaixo para responder à pergunta.
Cite as fontes ao responder. Se a informação não estiver nos documentos, diga isso explicitamente.

DOCUMENTOS RECUPERADOS (ordenados por relevância via RAG-Fusion):
{contexto}

PERGUNTA: {query}

RESPOSTA FUNDAMENTADA:"""
    
    response = llm.invoke(PROMPT_GERACAO)
    return response.content

# Gerar resposta
print('Gerando resposta com Llama 3.1 8B a partir dos documentos RAG-Fusion...')
resposta = gerar_resposta(query_complexa, resultado['fused_docs'], top_k=5)

print(f'\n=== RESPOSTA FINAL (RAG-Fusion + Llama 3.1) ===')
print(resposta)
print('='*60)

## 8. Visualização do Pipeline RAG-Fusion

In [ ]:
# Visualização dos scores RRF dos top documentos
fig, ax = plt.subplots(figsize=(12, 5))

top_docs = resultado['fused_docs'][:10]
labels = [f'Doc {i+1}\n(RRF={d["rrf_score"]:.4f})' for i, d in enumerate(top_docs)]
scores = [d['rrf_score'] for d in top_docs]

bars = ax.bar(range(len(top_docs)), scores, color='#3498db', alpha=0.8)
ax.set_title(f'RAG-Fusion — Top {len(top_docs)} Documentos por Score RRF\n'
             f'(N_queries={resultado["stats"]["n_queries"]}, k=60)', fontsize=12)
ax.set_xticks(range(len(top_docs)))
ax.set_xticklabels([f'Doc {i+1}' for i in range(len(top_docs))], rotation=45)
ax.set_ylabel('Score RRF')
ax.grid(axis='y', alpha=0.3)

# Adicionar valores nas barras
for i, (bar, score) in enumerate(zip(bars, scores)):
    ax.text(bar.get_x() + bar.get_width()/2, score + 0.0001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('rag_fusion_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico salvo como rag_fusion_scores.png')

In [ ]:
print('=== CHECKLIST DE ENTREGA — LAB 3 ===')
checklist = [
    'Sub-queries geradas com vLLM (N=3)',
    'Retrieval paralelo com asyncio.gather() executado',
    'RRF implementado manualmente com fórmula visível',
    'Exemplo numérico do RRF documentado e executado',
    'Geração final com Llama 3.1 a partir do contexto fundido',
    'Gráfico de scores RRF gerado'
]
for item in checklist:
    print(f'  [ ] {item}')
print('\n✅ Lab 3 concluído! Prossiga para o LAB 4 — Benchmark.')